In [25]:
import laspy
import glob
from os.path import join, basename, exists, dirname, abspath
import json
import requests
from glob import glob
import pdal



#### After downloading a text file of LAZ tiles within AOI via https://apps.nationalmap.gov/lidar-explorer/#/ , download tiles: 

In [19]:
def download_file(url):
    response = requests.get(url)
    if "content-disposition" in response.headers:
        content_disposition = response.headers["content-disposition"]
        filename = content_disposition.split("filename=")[1]
    else:
        filename = url.split("/")[-1]
    with open(filename, mode="wb") as file:
        file.write(response.content)
    print(f"Downloaded file {filename}")


url_list = "C:/Users/RDCRLSMC/Desktop/camas_3dep/downloadlist_pcs.txt"
# Open and read the file
with open(url_list, 'r') as file:
    urls = [line.strip() for line in file if line.strip()]  # Read and strip each line


for url in urls:
    download_file(url)  

#### Create definition for merging tiles by collection:

In [42]:
# Define the directory where your files are located
dir = abspath("C:/Users/RDCRLSMC/Desktop/camas_3dep/laz_tiles")

#list of FEMA collection
fema_fps = glob.glob(os.path.join(dir, '*FEMA*.laz'))
print(fema_fps)
print('----------------------------------------------------')

#SID collection 

sid_fps = glob.glob(os.path.join(dir, "*Southern*.laz"))
print(sid_fps)

mosaic_FEMA = join(dir, 'FEMA_merge.laz')
mosaic_SID = join(dir, 'SID_merge.laz')

['C:\\Users\\RDCRLSMC\\Desktop\\camas_3dep\\laz_tiles\\USGS_LPC_ID_FEMAHQ_2018_D18_w1542n2417.laz', 'C:\\Users\\RDCRLSMC\\Desktop\\camas_3dep\\laz_tiles\\USGS_LPC_ID_FEMAHQ_2018_D18_w1545n2415.laz', 'C:\\Users\\RDCRLSMC\\Desktop\\camas_3dep\\laz_tiles\\USGS_LPC_ID_FEMAHQ_2018_D18_w1545n2416.laz', 'C:\\Users\\RDCRLSMC\\Desktop\\camas_3dep\\laz_tiles\\USGS_LPC_ID_FEMAHQ_2018_D18_w1545n2417.laz', 'C:\\Users\\RDCRLSMC\\Desktop\\camas_3dep\\laz_tiles\\USGS_LPC_ID_FEMAHQ_2018_D18_w1546n2415.laz', 'C:\\Users\\RDCRLSMC\\Desktop\\camas_3dep\\laz_tiles\\USGS_LPC_ID_FEMAHQ_2018_D18_w1546n2416.laz', 'C:\\Users\\RDCRLSMC\\Desktop\\camas_3dep\\laz_tiles\\USGS_LPC_ID_FEMAHQ_2018_D18_w1546n2417.laz', 'C:\\Users\\RDCRLSMC\\Desktop\\camas_3dep\\laz_tiles\\USGS_LPC_ID_FEMAHQ_2018_D18_w1547n2416.laz', 'C:\\Users\\RDCRLSMC\\Desktop\\camas_3dep\\laz_tiles\\USGS_LPC_ID_FEMAHQ_2018_D18_w1547n2417.laz', 'C:\\Users\\RDCRLSMC\\Desktop\\camas_3dep\\laz_tiles\\USGS_LPC_ID_FEMAHQ_2018_D18_w1547n2418.laz', 'C:\\User

In [43]:
# Define a function to create and execute the PDAL pipeline
def run_pdal_pipeline(input_files, mosaic_fp):
    pipeline = {
        "pipeline": [
            # Create the readers dynamically for each file
            *[{"type": "readers.las", "filename": fp} for fp in input_files],
            {"type": "filters.merge"},  # Merge all the files
            {"type": "writers.las", "filename": mosaic_fp}  # Output to the mosaic file
        ]
    }

    # Convert the pipeline dictionary to a JSON string
    pipeline_str = json.dumps(pipeline)

    # Create the PDAL pipeline object
    p = pdal.Pipeline(pipeline_str)

    # Execute the pipeline
    count = p.execute()

    # Print out the result
    print(f"Merged {len(input_files)} files into '{mosaic_fp}'. Processed {count} points.")


# Run the pipeline for both FEMA and SID datasets
run_pdal_pipeline(fema_fps, mosaic_FEMA)
##run_pdal_pipeline(sid_fps, mosaic_SID)
    ##produces error: "MemoryError: bad allocation". Needs to be merged on borah

Merged 39 files into 'C:\Users\RDCRLSMC\Desktop\camas_3dep\laz_tiles\FEMA_merge.laz'. Processed 211262913 points.


MemoryError: bad allocation

#### Merge point clouds by collection:

#### Create vector object of extents:

In [14]:
!cd "C:/Users/RDCRLSMC/Desktop/camas_3dep/laz_tiles/SouthernID_clip" && pdal tindex create --tindex boundary.sqlite --filespec SouthernID_reproject.laz -f SQLite
!cd "C:/Users/RDCRLSMC/Desktop/camas_3dep/laz_tiles/FEMA" && pdal tindex create --tindex boundary.sqlite --filespec reproject.laz -f SQLite


PDAL: bad allocation



#### Clip  each point cloud to a shapefile of Camas. I've donw this by creating a geoJSON of the kml in QGIS.

In [44]:
def clip_point_cloud(input_file, polygon_file, output_file):
    pipeline = [
        {
            "type": "readers.las",
            "filename": input_file
        },
        {
            "type": "filters.crop",
            "polygon": polygon_file
        },
        {
            "type": "writers.las",
            "filename": output_file
        }
    ]

    pdal_pipeline = pdal.Pipeline(json.dumps(pipeline))
    pdal_pipeline.execute()

input_las = "your_point_cloud.las"
polygon_geojson = "your_polygon.geojson"
output_las = "clipped_point_cloud.las"

In [ ]:
clip_point_cloud(input_las, polygon_geojson, output_las)

#### And rasterize: